In [ ]:
Oracle AI Data Platform v1.0

Copyright © 2025, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# MongoDB Atlas Connector Samples

Read-only ingestion samples for MongoDB Atlas using the MongoDB Spark Connector, Spark format `mongodb`. Replace all placeholders before running a sample.

## Prerequisites

1. **Connector jars.** Run the *One-Time Setup* cell below: it downloads the five jars, connector 10.7.0 and driver 5.1.4, from Maven Central into a workspace folder and checks each SHA-256. You can also download them yourself from the links in this folder's `README.md`. Install each jar from the cluster's **Library** tab with **Install Library** → **Workspace**, one at a time, then **Actions → Restart**. Loading the jars at runtime with `addJar` is not enough on AIDP: executor tasks fail with `UnknownReason`.
2. **Credential.** In the Credential Store, create a **Secret Token** credential with key `uri`. Its value is the cluster's connection string for a read-only database user, from Atlas **Connect → Drivers**. Never type it into a cell. It looks like:
   ```
   mongodb+srv://<user>:<password>@<cluster>.mongodb.net/
   ```
3. **Network.** Add the cluster's outbound IP to the Atlas project's **IP access list**, and check it again after a cluster restart. A blocked IP fails after the server-selection timeout with `SSLException: Received fatal alert: internal_error`.

## One-Time Setup: Download the Connector Jars

Downloads the five jars into a `jars` folder inside `WORKSPACE_FOLDER` and checks each against its pinned SHA-256; a file that does not match is deleted. Files already there and matching are kept. Then install them from that `jars` folder as cluster libraries and restart the cluster, as in Prerequisite 1. Skip this cell once the jars are installed.

In [ ]:
import hashlib
import os
import urllib.request

WORKSPACE_FOLDER = "<WORKSPACE_FOLDER>"  # e.g. the folder this notebook is in, under /Workspace/
JAR_DIR = os.path.join(WORKSPACE_FOLDER, "jars")
MAVEN = "https://repo1.maven.org/maven2/org/mongodb/"
JARS = {
    "spark/mongo-spark-connector_2.12/10.7.0/mongo-spark-connector_2.12-10.7.0.jar":
        "1b0908775a41d72621944a43e36ed83df4dbaff5bf8581811f0b5e9eabeb7cbe",
    "mongodb-driver-sync/5.1.4/mongodb-driver-sync-5.1.4.jar":
        "341880078296edd762756440e9d6b1d6d2bf4d6b91975c2638e3da611f28b1c2",
    "mongodb-driver-core/5.1.4/mongodb-driver-core-5.1.4.jar":
        "ae3dbfd439d5afe9e0d6abbd61ef27d858ca2d38083bb7361f69e243aff31dec",
    "bson/5.1.4/bson-5.1.4.jar":
        "bba556a8acd4e87545c1b9a1cb25c12ce9587ed5bf01685f236c5d92abf1e676",
    "bson-record-codec/5.1.4/bson-record-codec-5.1.4.jar":
        "698b2b9a10fdd49a3ed99ad2b7bcc8e797639c8c4a5c974de7f6fc8654bda655",
}


def sha256_of(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


os.makedirs(JAR_DIR, exist_ok=True)
for path, expected in JARS.items():
    target = os.path.join(JAR_DIR, path.rsplit("/", 1)[1])
    if not os.path.exists(target) or sha256_of(target) != expected:
        urllib.request.urlretrieve(MAVEN + path, target)
        if sha256_of(target) != expected:
            os.remove(target)
            raise ValueError("SHA-256 mismatch, file removed: " + target)
    print(target)

## Configuration

In [ ]:
CREDENTIAL_NAME = "<CREDENTIAL_NAME>"  # Secret Token credential with key "uri"
DATABASE = "<DATABASE>"
COLLECTION = "<COLLECTION>"
TARGET = "<CATALOG>.<SCHEMA>.<TABLE_NAME>"
WATERMARK_FIELD = "<DATE_FIELD>"  # must hold BSON Dates, not strings
STRING_FIELDS = []                # top-level fields whose type differs between documents
SAMPLE_SIZE = 10000               # documents sampled to infer the schema
OVERLAP_SECONDS = 300

## Connection URI

Reads the connection string from the Credential Store. AIDP executors cannot resolve an SRV connection string, one starting with `mongodb+srv`: the lookup fails with *Failed looking up TXT record*. So the SRV and TXT records are looked up here, on the driver, and Spark gets an equivalent standard connection string that lists the hosts. It is rebuilt on every run, so Atlas host changes are picked up.

In [ ]:
def resolve_srv(uri):
    if not uri.startswith("mongodb+srv://"):
        return uri
    rest, _, query = uri[len("mongodb+srv://"):].partition("?")
    authority, _, path = rest.partition("/")
    userinfo, _, host = authority.rpartition("@")
    env = spark._jvm.java.util.Hashtable()
    env.put("java.naming.factory.initial", "com.sun.jndi.dns.DnsContextFactory")
    dns = spark._jvm.javax.naming.directory.InitialDirContext(env)

    def records(name, kind):
        kinds = spark.sparkContext._gateway.new_array(spark._jvm.java.lang.String, 1)
        kinds[0] = kind
        attr = dns.getAttributes(name, kinds).get(kind)
        return [str(attr.get(i)) for i in range(attr.size())] if attr is not None else []

    parent = host.split(".", 1)[1].lower()
    hosts = []
    for record in records("_mongodb._tcp." + host, "SRV"):
        _, _, port, target = record.split()
        target = target.rstrip(".")
        if not target.lower().endswith("." + parent):  # same rule as the driver: stay in the cluster's domain
            raise ValueError("SRV record outside the cluster's domain")
        hosts.append(target + ":" + port)
    if not hosts:
        raise ValueError("SRV lookup returned no hosts")
    options = {}
    for record in records(host, "TXT"):
        for pair in record.strip('"').split("&"):
            key, _, value = pair.partition("=")
            if key in ("authSource", "replicaSet", "loadBalanced"):
                options[key] = value
    options.update(pair.partition("=")[::2] for pair in query.split("&") if pair)
    if not any(key.lower() in ("tls", "ssl") for key in options):
        options["tls"] = "true"
    options.setdefault("serverSelectionTimeoutMS", "10000")  # fail fast instead of the driver's 30 s
    return "mongodb://{}{}/{}?{}".format(
        userinfo + "@" if userinfo else "", ",".join(hosts), path,
        "&".join(key + "=" + value for key, value in options.items()))


uri = resolve_srv(aidputils.secrets.get(name=CREDENTIAL_NAME, key="uri").strip())
print("connection URI resolved (not shown)")

## Ingestion Sample

Reads the collection with a schema inferred from a sample. The first `show` also proves network access, the IP access list and authentication.

In [ ]:
mongodb_df = spark.read.format("mongodb") \
    .option("connection.uri", uri) \
    .option("database", DATABASE) \
    .option("collection", COLLECTION) \
    .option("sampleSize", SAMPLE_SIZE) \
    .load()

mongodb_df.show()

## Read with a Widened Schema

The inferred schema only fits the sampled documents: a Decimal128 wider than the sampled precision is read as `null` with no error. Widening every decimal to precision 38 with a scale of at least 10, and every int to a long, avoids that. A field whose *type* varies between documents fails the read with `UnknownReason`; list it in `STRING_FIELDS` to read it as text.

On AIDP, if even a one-row read fails with `UnknownReason`, the executors are missing the jars instead: check Prerequisite 1.

In [ ]:
import re
from pyspark.sql.types import StructType


def widen(t):
    if isinstance(t, str):
        if t == "integer":
            return "long"
        match = re.fullmatch(r"decimal\((\d+),\s*(\d+)\)", t)
        return "decimal(38,{})".format(max(int(match.group(2)), 10)) if match else t
    if t["type"] == "struct":
        return {**t, "fields": [{**f, "type": widen(f["type"]), "nullable": True} for f in t["fields"]]}
    if t["type"] == "array":
        return {**t, "elementType": widen(t["elementType"]), "containsNull": True}
    if t["type"] == "map":
        return {**t, "valueType": widen(t["valueType"]), "valueContainsNull": True}
    return t


wide = widen(mongodb_df.schema.jsonValue())
for field in wide["fields"]:
    if field["name"] in STRING_FIELDS:
        field["type"] = "string"
wide_schema = StructType.fromJson(wide)

mongodb_df_wide = spark.read.format("mongodb") \
    .option("connection.uri", uri) \
    .option("database", DATABASE) \
    .option("collection", COLLECTION) \
    .schema(wide_schema) \
    .load()

mongodb_df_wide.show()

## Pushdown Pipeline Sample

`aggregation.pipeline` runs an aggregation pipeline in MongoDB before the data reaches Spark.

In [ ]:
mongodb_df_pushdown = spark.read.format("mongodb") \
    .option("connection.uri", uri) \
    .option("database", DATABASE) \
    .option("collection", COLLECTION) \
    .option("aggregation.pipeline", '[{"$match": {"<FIELD_NAME>": "<VALUE>"}}]') \
    .load()

mongodb_df_pushdown.show()

## Incremental Load into Delta

The first run loads everything into `TARGET`. Later runs read only documents whose `WATERMARK_FIELD` is newer than the newest one already in the table, minus `OVERLAP_SECONDS`. That filter runs in MongoDB. The documents are merged into the table on `_id`, which also removes the overlap's duplicates. Later runs reuse the table's schema. The watermark is read as epoch microseconds because a collected TIMESTAMP comes back in the Python process's local timezone. A watermark does not see deletes.

In [ ]:
import json
from datetime import datetime, timedelta, timezone


def as_bson_date(value):
    value = value.astimezone(timezone.utc)
    return {"$date": value.strftime("%Y-%m-%dT%H:%M:%S.") + "%03dZ" % (value.microsecond // 1000)}


since, schema = None, wide_schema
if spark.catalog.tableExists(TARGET):
    schema = spark.table(TARGET).schema
    micros = spark.sql(f"SELECT unix_micros(max(`{WATERMARK_FIELD}`)) AS m FROM {TARGET}").first()["m"]
    if micros is not None:
        since = datetime(1970, 1, 1, tzinfo=timezone.utc) + timedelta(microseconds=micros)

bounds = {"$lte": as_bson_date(datetime.now(timezone.utc))}
if since is not None:
    bounds["$gte"] = as_bson_date(since - timedelta(seconds=OVERLAP_SECONDS))

mongodb_df_incremental = spark.read.format("mongodb") \
    .option("connection.uri", uri) \
    .option("database", DATABASE) \
    .option("collection", COLLECTION) \
    .option("aggregation.pipeline", json.dumps([{"$match": {WATERMARK_FIELD: bounds}}])) \
    .schema(schema) \
    .load()

if not spark.catalog.tableExists(TARGET):
    mongodb_df_incremental.write.format("delta").saveAsTable(TARGET)
else:
    mongodb_df_incremental.createOrReplaceTempView("incoming_document")
    spark.sql(
        f"MERGE INTO {TARGET} t USING incoming_document s ON t._id = s._id "
        "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *"
    )

print("documents in target:", spark.table(TARGET).count())

## Connector Options

| Parameter name | Valid values | Mandatory | Description |
| --- | --- | --- | --- |
| `connection.uri` | Standard `mongodb` connection string | Y | Connection string. On AIDP, resolve an SRV string, one starting with `mongodb+srv`, on the driver first; see *Connection URI*. |
| `database` | Valid database name | Y | Database to read from. |
| `collection` | Valid collection name | Y | Collection to read from. |
| `aggregation.pipeline` | JSON array of pipeline stages | N | Aggregation pipeline run in MongoDB before the read, for example a `$match` filter. |
| `sampleSize` | Positive integer | N | Number of documents sampled to infer the schema. Ignored when a schema is passed to the reader. |